# FedSafe-Fuse — SACP coverage with Clopper–Pearson CIs (m3)
Inference only (no training), on the **seed-42** checkpoints that produced the coverage tables. Run top-to-bottom (~2–3 min). Reports each coverage with a pixel-level **Clopper–Pearson 95% CI** and an image-level **bootstrap 95% CI** (which accounts for within-image pixel correlation and small #slices). The MedMNIST cell self-checks the recomputed rates against the published table.

In [ ]:
# SACP coverage + Clopper-Pearson CIs (m3) — inference only, seed-42 checkpoints
import os, time, json, csv
import numpy as np
import torch
import cv2
from scipy.stats import beta
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def clopper_pearson(k, n, alpha=0.05):
    """Exact binomial (Clopper-Pearson) CI for k successes in n trials."""
    if n == 0:
        return (float('nan'), float('nan'))
    lo = 0.0 if k == 0 else float(beta.ppf(alpha / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(beta.ppf(1 - alpha / 2, k + 1, n - k))
    return lo, hi

def image_bootstrap_ci(per_image_cov, n_boot=10000, alpha=0.05, seed=0):
    """Percentile bootstrap CI for mean coverage, resampling IMAGES (accounts for
    within-image pixel correlation; meaningful when #images is small, e.g. BraTS)."""
    a = np.asarray(per_image_cov, dtype=np.float64)
    if len(a) == 0:
        return (float('nan'), float('nan'))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n_boot, len(a)))
    means = a[idx].mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

CANNY_LOW, CANNY_HIGH = 50, 100
DIL = np.ones((3, 3), np.uint8)
ALPHAS = [0.05, 0.10, 0.15]
BATCH = 16


In [ ]:
# Mount Drive + mirror checkpoints, calibration data, partitions
from google.colab import drive
import shutil
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL_ROOT = '/content/local_fedsafe'
for sub in ['medmnist', 'partitions', 'brats/processed', 'checkpoints', 'results']:
    os.makedirs(f'{LOCAL_ROOT}/{sub}', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

for fn in ['calibration_manifest.json', 'medmnist_K3.json', 'brats_K3.json']:
    s = f'{DRIVE_ROOT}/partitions/{fn}'; d = f'{LOCAL_ROOT}/partitions/{fn}'
    if os.path.exists(s) and not os.path.exists(d): shutil.copy(s, d)
for fn in ['fedsafe_fipca_r50.pt', 'ifcnn_fedavg_brats_r50.pt']:
    s = f'{DRIVE_ROOT}/checkpoints/{fn}'; d = f'{LOCAL_ROOT}/checkpoints/{fn}'
    if os.path.exists(s) and not os.path.exists(d): shutil.copy(s, d)
for fn in ['train_paired.npz', 'test_paired.npz']:
    s = f'{DRIVE_ROOT}/medmnist/{fn}'; d = f'{LOCAL_ROOT}/medmnist/{fn}'
    if not os.path.exists(d):
        print('copying', fn); shutil.copy(s, d)
# BraTS calibration case files (first 12 calibration cases, as in 10b)
with open(f'{LOCAL_ROOT}/partitions/brats_K3.json') as f: _bpart = json.load(f)
brats_calib_case_list = _bpart['calibration_cases'][:12]
for cid in brats_calib_case_list:
    s = f'{DRIVE_ROOT}/brats/processed/{cid}.npz'; d = f'{LOCAL_ROOT}/brats/processed/{cid}.npz'
    if os.path.exists(s) and not os.path.exists(d): shutil.copy(s, d)
PROJECT_DRIVE = LOCAL_ROOT
print('mirror ready')


## Model definitions (verbatim from 03_fedtrain)

In [ ]:
# FedSafe-Fuse model
"""FedSafe-Fuse: dual MobileNetV3-Small encoders + 2-layer Transformer + conv decoder.

Per the project proposal:
    - Two modality-specific encoders, each MobileNetV3-Small (~2.5M params total each)
    - Shared 2-layer Transformer module (2 heads, embed dim 128) for cross-modal attention
    - Convolutional decoder
    - Target ~8M params

Actual realised parameter count depends on decoder width; default config lands at ~5M.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small


class ModalityEncoder(nn.Module):
    """MobileNetV3-Small features adapted to 1-channel input.

    MobileNetV3-Small downsamples by 32x, so a 128x128 input yields 4x4 features.
    We bilinearly upsample to 8x8 to give the cross-modal Transformer 64 tokens
    per modality (matching the proposal). The upsample is parameter-free.
    """

    def __init__(self, embed_dim: int = 128):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        # Replace first conv to accept a single channel (medical grayscale)
        base.features[0][0] = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.features = base.features  # (B, 576, 4, 4) for 128x128 input
        self.proj = nn.Conv2d(576, embed_dim, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        h = self.proj(h)
        return self.upsample(h)  # (B, embed_dim, 8, 8)


class CrossModalTransformer(nn.Module):
    """2-layer Transformer encoder over concatenated MRI/PET tokens.

    Input feature maps are flattened to 64 spatial tokens per modality, given a
    learned modality embedding, concatenated to 128 tokens, given a learned
    positional embedding, and passed through `num_layers` transformer encoder layers.
    Output is averaged across the two modality halves and reshaped back to a
    spatial feature map for the decoder.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 2,
        ff_dim: int = 256,
        dropout: float = 0.1,
        spatial_tokens_per_modality: int = 64,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        total_tokens = spatial_tokens_per_modality * 2
        self.pos_embed = nn.Parameter(torch.zeros(1, total_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.mod_embed = nn.Embedding(2, embed_dim)  # 0 = MRI, 1 = PET
        self.spatial_tokens = spatial_tokens_per_modality

    def forward(self, mri_feat: torch.Tensor, pet_feat: torch.Tensor) -> torch.Tensor:
        # mri_feat, pet_feat: (B, C, H, W) with H*W = self.spatial_tokens
        B, C, H, W = mri_feat.shape
        mri_tokens = mri_feat.flatten(2).transpose(1, 2)  # (B, HW, C)
        pet_tokens = pet_feat.flatten(2).transpose(1, 2)
        mri_tokens = mri_tokens + self.mod_embed.weight[0]
        pet_tokens = pet_tokens + self.mod_embed.weight[1]
        tokens = torch.cat([mri_tokens, pet_tokens], dim=1) + self.pos_embed
        out = self.transformer(tokens)
        mri_out = out[:, : self.spatial_tokens].transpose(1, 2).reshape(B, C, H, W)
        pet_out = out[:, self.spatial_tokens :].transpose(1, 2).reshape(B, C, H, W)
        return 0.5 * (mri_out + pet_out)


class FusionDecoder(nn.Module):
    """Progressive bilinear upsample + conv decoder: 8x8 -> 16 -> 32 -> 64 -> 128."""

    def __init__(self, embed_dim: int = 128, base_ch: int = 256):
        super().__init__()
        c0, c1, c2, c3 = base_ch, base_ch, base_ch // 2, base_ch // 4
        self.up1 = self._block(embed_dim, c0)
        self.up2 = self._block(c0, c1)
        self.up3 = self._block(c1, c2)
        self.up4 = self._block(c2, c3)
        self.final = nn.Conv2d(c3, 1, kernel_size=1)

    @staticmethod
    def _block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up1(x)
        h = self.up2(h)
        h = self.up3(h)
        h = self.up4(h)
        return torch.sigmoid(self.final(h))


class FedSafeFuse(nn.Module):
    """End-to-end FedSafe-Fuse model. Inputs: two (B, 1, 128, 128) tensors. Output: (B, 1, 128, 128)."""

    def __init__(
        self,
        embed_dim: int = 128,
        transformer_layers: int = 2,
        transformer_heads: int = 2,
        transformer_ff: int = 256,
        decoder_base_ch: int = 256,
    ):
        super().__init__()
        self.mri_encoder = ModalityEncoder(embed_dim)
        self.pet_encoder = ModalityEncoder(embed_dim)
        self.fusion = CrossModalTransformer(
            embed_dim=embed_dim,
            num_layers=transformer_layers,
            num_heads=transformer_heads,
            ff_dim=transformer_ff,
        )
        self.decoder = FusionDecoder(embed_dim=embed_dim, base_ch=decoder_base_ch)

    def forward(self, mri: torch.Tensor, pet: torch.Tensor) -> torch.Tensor:
        mri_feat = self.mri_encoder(mri)
        pet_feat = self.pet_encoder(pet)
        fused_feat = self.fusion(mri_feat, pet_feat)
        return self.decoder(fused_feat)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [ ]:
# IFCNN baseline
"""IFCNN baseline (Zhang et al. 2021).

A general image fusion framework based on a two-stream CNN with element-wise fusion
followed by reconstruction. Simplified from the original paper for our 128x128
single-channel fusion setting. Used as Baseline 2 in the FedSafe-Fuse evaluation.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class IFCNN(nn.Module):
    """Two-stream conv + element-wise fusion + conv reconstruction."""

    FUSION_MODES = ("max", "mean", "sum")

    def __init__(self, base_ch: int = 64, fusion: str = "max"):
        super().__init__()
        assert fusion in self.FUSION_MODES, f"fusion must be one of {self.FUSION_MODES}"
        self.fusion_mode = fusion
        self.stream1 = self._make_stream(base_ch)
        self.stream2 = self._make_stream(base_ch)
        self.recon = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=1),
            nn.BatchNorm2d(base_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch // 2, 1, kernel_size=1),
        )

    @staticmethod
    def _make_stream(c: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        f1 = self.stream1(x1)
        f2 = self.stream2(x2)
        if self.fusion_mode == "max":
            fused = torch.maximum(f1, f2)
        elif self.fusion_mode == "mean":
            fused = 0.5 * (f1 + f2)
        else:
            fused = f1 + f2
        return torch.sigmoid(self.recon(fused))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



## Shared coverage / CI helpers

In [ ]:
# Coverage evaluator: given a model and a list of (mri,pet) batches provider,
# returns per-region pooled (k,n) and per-image coverage lists for each alpha.
def eval_coverage(get_pair, ids, model, thr):
    ki = {a: 0 for a in ALPHAS}; ni = 0
    kb = {a: 0 for a in ALPHAS}; nb = 0
    per_img = {a: {'interior': [], 'boundary': []} for a in ALPHAS}
    with torch.no_grad():
        for i in range(0, len(ids), BATCH):
            b = ids[i:i + BATCH]
            mri, pet = get_pair(b)
            fused = model(mri, pet); ref = 0.5 * (mri + pet)
            res = (fused - ref).abs().cpu().numpy()
            refb = (ref.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            for k in range(len(b)):
                em = cv2.dilate(cv2.Canny(refb[k, 0], CANNY_LOW, CANNY_HIGH), DIL, iterations=1) > 0
                r = res[k, 0]; ri = r[~em]; rb = r[em]
                ni += len(ri); nb += len(rb)
                for a in ALPHAS:
                    ci = int((ri <= thr[a]['interior']).sum())
                    cb = int((rb <= thr[a]['boundary']).sum())
                    ki[a] += ci; kb[a] += cb
                    if len(ri): per_img[a]['interior'].append(ci / len(ri))
                    if len(rb): per_img[a]['boundary'].append(cb / len(rb))
    return ki, ni, kb, nb, per_img

def calib_thresholds(get_pair, ids, model):
    ic, bc = [], []
    with torch.no_grad():
        for i in range(0, len(ids), BATCH):
            b = ids[i:i + BATCH]
            mri, pet = get_pair(b)
            fused = model(mri, pet); ref = 0.5 * (mri + pet)
            res = (fused - ref).abs().cpu().numpy()
            refb = (ref.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            for k in range(len(b)):
                em = cv2.dilate(cv2.Canny(refb[k, 0], CANNY_LOW, CANNY_HIGH), DIL, iterations=1) > 0
                r = res[k, 0]; ic.append(r[~em].astype(np.float32)); bc.append(r[em].astype(np.float32))
    ic = np.concatenate(ic); bc = np.concatenate(bc)
    return {a: {'interior': float(np.quantile(ic, 1 - a)),
                'boundary': float(np.quantile(bc, 1 - a))} for a in ALPHAS}

def build_rows(tag, thr, ki, ni, kb, nb, per_img):
    rows = []
    for a in ALPHAS:
        icov = ki[a] / max(ni, 1); bcov = kb[a] / max(nb, 1)
        il, ih = clopper_pearson(ki[a], ni); bl, bh = clopper_pearson(kb[a], nb)
        ibl, ibh = image_bootstrap_ci(per_img[a]['interior'])
        bbl, bbh = image_bootstrap_ci(per_img[a]['boundary'])
        rows.append(dict(alpha=a, target=round(1 - a, 2),
            interior_coverage=round(icov, 4), interior_cp_lo=round(il, 4), interior_cp_hi=round(ih, 4),
            interior_img_lo=round(ibl, 4), interior_img_hi=round(ibh, 4),
            interior_width=round(2 * thr[a]['interior'], 4),
            boundary_coverage=round(bcov, 4), boundary_cp_lo=round(bl, 4), boundary_cp_hi=round(bh, 4),
            boundary_img_lo=round(bbl, 4), boundary_img_hi=round(bbh, 4),
            boundary_width=round(2 * thr[a]['boundary'], 4),
            n_interior_px=ni, n_boundary_px=nb, n_images=len(per_img[a]['interior'])))
    return rows

def save_and_show(tag, rows, path, drive_path):
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    shutil.copy(path, drive_path)
    print(f'\n[{tag}] saved {path} (+Drive)')
    for r in rows:
        print(f"  a={r['alpha']} target={r['target']:.2f} | "
              f"INT {r['interior_coverage']:.3f} CP[{r['interior_cp_lo']:.3f},{r['interior_cp_hi']:.3f}] "
              f"img[{r['interior_img_lo']:.3f},{r['interior_img_hi']:.3f}] | "
              f"BND {r['boundary_coverage']:.3f} CP[{r['boundary_cp_lo']:.3f},{r['boundary_cp_hi']:.3f}] "
              f"img[{r['boundary_img_lo']:.3f},{r['boundary_img_hi']:.3f}]")


## MedMNIST

In [ ]:
# ===== MedMNIST coverage + CP (FROP rank-32 seed-42 checkpoint) =====
with open(f'{PROJECT_DRIVE}/partitions/calibration_manifest.json') as f: manifest = json.load(f)
calib_indices = manifest['medmnist']['calibration_indices']
with np.load(f'{PROJECT_DRIVE}/medmnist/train_paired.npz') as d:
    tr_mri, tr_pet = d['mri'], d['pet']
with np.load(f'{PROJECT_DRIVE}/medmnist/test_paired.npz') as d:
    te_mri, te_pet = d['mri'], d['pet']
med_model = FedSafeFuse().to(DEVICE)
ck = torch.load(f'{PROJECT_DRIVE}/checkpoints/fedsafe_fipca_r50.pt', map_location=DEVICE)
med_model.load_state_dict(ck['state_dict']); med_model.eval()
print('MedMNIST checkpoint round', ck['round'], 'tag', ck['tag'])

def med_train_pair(b):
    return (torch.from_numpy(tr_mri[b]).unsqueeze(1).to(DEVICE),
            torch.from_numpy(tr_pet[b]).unsqueeze(1).to(DEVICE))
def med_test_pair(b):
    return (torch.from_numpy(te_mri[b]).unsqueeze(1).to(DEVICE),
            torch.from_numpy(te_pet[b]).unsqueeze(1).to(DEVICE))

med_thr = calib_thresholds(med_train_pair, list(calib_indices), med_model)
ki, ni, kb, nb, per_img = eval_coverage(med_test_pair, list(range(256)), med_model, med_thr)
med_rows = build_rows('medmnist', med_thr, ki, ni, kb, nb, per_img)
save_and_show('MedMNIST', med_rows, f'{PROJECT_DRIVE}/results/coverage_table_cp.csv',
              f'{DRIVE_ROOT}/results/coverage_table_cp.csv')
# self-check vs published coverage_table.csv
PUB = {0.05: (0.9692, 0.9599), 0.10: (0.9272, 0.9163), 0.15: (0.8821, 0.8708)}
print('  self-check vs published rates (interior, boundary):')
for r in med_rows:
    pi, pb = PUB[r['alpha']]
    ok = abs(r['interior_coverage'] - pi) < 0.005 and abs(r['boundary_coverage'] - pb) < 0.005
    print(f"    a={r['alpha']}: {r['interior_coverage']:.4f}/{r['boundary_coverage']:.4f} "
          f"vs {pi}/{pb} {'OK' if ok else '*** CHECK ***'}")


## BraTS

In [ ]:
# ===== BraTS coverage + CP (IFCNN seed-42 checkpoint) =====
from torch.utils.data import Dataset
class BraTSCaseDataset(Dataset):
    def __init__(self, case_ids, root, max_slices=None, rng_seed=42):
        t1s, t2s = [], []
        for cid in case_ids:
            p = f'{root}/brats/processed/{cid}.npz'
            if not os.path.exists(p): continue
            with np.load(p) as d:
                t1s.append(d['t1'].astype(np.float32)); t2s.append(d['t2'].astype(np.float32))
        self.t1 = np.concatenate(t1s, 0); self.t2 = np.concatenate(t2s, 0)
        if max_slices and len(self.t1) > max_slices:
            sel = np.random.default_rng(rng_seed).choice(len(self.t1), max_slices, replace=False)
            self.t1, self.t2 = self.t1[sel], self.t2[sel]
    def __len__(self): return len(self.t1)

# rebuild the exact seed-42 eval set (12 calib cases, max_slices=512), then split half/half
eval_ds = BraTSCaseDataset(brats_calib_case_list, PROJECT_DRIVE, max_slices=512, rng_seed=42)
n = len(eval_ds); half = n // 2
brats_model = IFCNN().to(DEVICE)
ck = torch.load(f'{PROJECT_DRIVE}/checkpoints/ifcnn_fedavg_brats_r50.pt', map_location=DEVICE)
brats_model.load_state_dict(ck['state_dict']); brats_model.eval()
print('BraTS eval slices:', n, '| calib/test split:', half, n - half)

def brats_pair(b):
    return (torch.stack([torch.from_numpy(eval_ds.t1[j]).unsqueeze(0) for j in b]).to(DEVICE),
            torch.stack([torch.from_numpy(eval_ds.t2[j]).unsqueeze(0) for j in b]).to(DEVICE))

br_thr = calib_thresholds(brats_pair, list(range(half)), brats_model)
ki, ni, kb, nb, per_img = eval_coverage(brats_pair, list(range(half, n)), brats_model, br_thr)
br_rows = build_rows('brats', br_thr, ki, ni, kb, nb, per_img)
save_and_show('BraTS', br_rows, f'{PROJECT_DRIVE}/results/coverage_table_brats_cp.csv',
              f'{DRIVE_ROOT}/results/coverage_table_brats_cp.csv')


## Export

In [ ]:
# ===== paste this back to Claude =====
print('===== PASTE THIS BACK TO CLAUDE =====')
print(json.dumps({'medmnist_coverage_cp': med_rows, 'brats_coverage_cp': br_rows}, indent=2))
